# 📋 Step 1: Sepsis Data Extraction from MIMIC-IV v3.1

## What This Notebook Does

This is the **first executable step** in our pipeline. It takes the raw MIMIC-IV data
(~70 GB of CSV files) and extracts a carefully defined **sepsis cohort** — the specific
group of ICU patients we will analyze.

### Why Extraction is Needed
MIMIC-IV contains data for ~300,000 patients. Most don't have sepsis. We need to:
1. Identify which patients meet our sepsis criteria
2. Pull their vital signs, lab results, and treatment records
3. Save everything as smaller, manageable files for the next steps

### Our Cohort Definition (Team-Agreed Clinical Criteria)
| Filter | Rule | Rationale |
|--------|------|-----------|
| First ICU stay | `ROW_NUMBER() OVER(PARTITION BY subject_id ORDER BY intime) = 1` | Prevent duplicate patients |
| Duration | 12 hours < stay < 10 days | Enough data for HMM, exclude very complex cases |
| Age | 18-65 years | Avoid pediatric and geriatric confounders |
| Sepsis | Antibiotics + cultures within 1 hour | Clinical suspicion of infection |

### Output Files
| File | Description | Approximate Size |
|------|------------|-----------------|
| `cohort.csv` | Patient list with demographics | ~200 KB |
| `vitals.csv` | Heart rate, BP, temp, SpO2, etc. | ~100-500 MB |
| `labs.csv` | Lactate, WBC, platelets, etc. | ~50-200 MB |
| Various treatment files | Antibiotics, vasopressors, IV fluids | ~50-200 MB |

## 📌 Cell 1: Configuration

Set file paths and parameters. **This is the only cell you need to modify** if running on a different machine.

In [1]:
import pandas as pd
import numpy as np
import os
import time

# =============================================================
# 🔧 USER CONFIGURATION — Only modify paths below!
# =============================================================

# MIMIC-IV data location on your machine
# The data comes in two modules: 'hosp' (hospital-wide) and 'icu' (ICU-specific)
# If all CSVs are in one folder, set both to the same path
DATA_DIR = "/Users/hoon/Documents/10_Classes/12_Bayesian Machine Learning with Generative AI Applications/10_Final_Project/01_Data"
ICU_DIR = DATA_DIR      # Contains: icustays.csv, chartevents.csv, inputevents.csv, etc.
HOSP_DIR = DATA_DIR     # Contains: patients.csv, admissions.csv, labevents.csv, etc.

# Where to save extracted data (will be created if it doesn't exist)
OUTPUT_DIR = "/Users/hoon/Documents/10_Classes/12_Bayesian Machine Learning with Generative AI Applications/10_Final_Project/02_Processed_Data"

# How many rows to read at a time from the largest files
# chartevents.csv is 41.94 GB — we MUST read it in chunks
CHUNK_SIZE = 1_000_000  # 1 million rows per chunk

# =============================================================
os.makedirs(OUTPUT_DIR, exist_ok=True)
start_time = time.time()
print(f"ICU dir:  {ICU_DIR}")
print(f"Hosp dir: {HOSP_DIR}")
print(f"Output:   {OUTPUT_DIR}")

ICU dir:  /Users/hoon/Documents/10_Classes/12_Bayesian Machine Learning with Generative AI Applications/10_Final_Project/01_Data
Hosp dir: /Users/hoon/Documents/10_Classes/12_Bayesian Machine Learning with Generative AI Applications/10_Final_Project/01_Data
Output:   /Users/hoon/Documents/10_Classes/12_Bayesian Machine Learning with Generative AI Applications/10_Final_Project/02_Processed_Data


## 📌 Cell 2: Item ID Lists for Sepsis Definition

MIMIC-IV uses numeric **item IDs** to identify what was measured or administered.
For example, item 220045 = Heart Rate, item 225798 = Vancomycin.

These IDs are defined in `d_items.csv`. Below we list every ID we need, organized by category.
These were determined by our team based on the MIMIC-IV documentation and clinical guidelines.

In [2]:
# =============================================================
# Culture tests (from procedureevents)
# =============================================================
CULTURE_ITEMIDS = [
    225401,  # Blood culture (blood culture)
    225437,  # CSF culture (CSF)
    225444,  # Pan culture (pan culture)
    225451,  # Sputum culture (sputum)
    225454,  # Urine culture (urine)
    225816,  # Wound culture (wound)
    225817,  # BAL culture (bronchoalveolar lavage)
    225818,  # Pleural culture (pleural fluid)
]

# =============================================================
# Antibiotics (from inputevents — IV administration records)
# =============================================================
ANTIBIOTIC_ITEMIDS = [
    225798,  # Vancomycin
    225842,  # Ampicillin
    225845,  # Azithromycin
    225847,  # Aztreonam
    225850,  # Cefazolin
    225851,  # Cefepime
    225855,  # Ceftriaxone
    225860,  # Clindamycin
    225879,  # Levofloxacin
    225881,  # Linezolid
    225883,  # Meropenem
    225884,  # Metronidazole
    225886,  # Moxifloxacin
    225892,  # Piperacillin
    225893,  # Piperacillin/Tazobactam (Zosyn)
    225899,  # Bactrim (SMX/TMP)
    225902,  # Tobramycin
    229061,  # Ertapenem sodium (Invanz)
]

# =============================================================
# Oxygen support (chartevents + procedureevents)
# =============================================================
INVASIVE_OXYGEN = [
    220339, 223849, 224700, 224829, 225308,
    225411, 225792, 226260, 228719, 229314,
]
NON_INVASIVE_OXYGEN = [225949, 225794, 227578]
ALL_OXYGEN = INVASIVE_OXYGEN + NON_INVASIVE_OXYGEN

# =============================================================
# Vasopressors (inputevents)
# =============================================================
VASOPRESSOR_ITEMIDS = [
    221289, 221653, 221662, 221749, 221906,
    221986, 222315, 229617, 229630, 229631,
    229632, 229709, 229764,
]

# =============================================================
# Vital signs + labs (chartevents)
# =============================================================
VITALS_LAB_ITEMIDS = [
    220045,  # Heart Rate
    220052,  # ABP Mean (MAP)
    220277,  # SpO2
    220546,  # WBC (from chartevents)
    225668,  # Lactic Acid
]

# =============================================================
# Additional vitals (compatible with original pipeline)
# =============================================================
ADDITIONAL_VITAL_ITEMIDS = {
    220045: 'Heart Rate',
    220050: 'ABP Systolic',
    220051: 'ABP Diastolic',
    220052: 'ABP Mean',
    225312: 'ART BP Systolic',
    220179: 'NIBP Systolic',
    220180: 'NIBP Diastolic',
    220181: 'NIBP Mean',
    223761: 'Temperature (F)',
    223762: 'Temperature (C)',
    220277: 'SpO2',
    220235: 'FiO2',
    220210: 'Respiratory Rate',
    224690: 'Respiratory Rate (Total)',
    220739: 'GCS - Eye Opening',
    223900: 'GCS - Verbal Response',
    223901: 'GCS - Motor Response',
    226559: 'Foley Output',
}

# Lab items (from labevents)
LAB_ITEMIDS = {
    50813: 'Lactate', 51652: 'Procalcitonin', 50889: 'C-Reactive Protein',
    51265: 'Platelet Count', 50885: 'Bilirubin Total', 50912: 'Creatinine',
    51301: 'White Blood Cells', 51300: 'WBC Count',
    51222: 'Hemoglobin', 51221: 'Hematocrit',
    51006: 'BUN', 50862: 'Albumin', 50931: 'Glucose',
    50971: 'Potassium', 50983: 'Sodium', 50902: 'Chloride',
    50820: 'pH', 50821: 'pO2', 50818: 'pCO2',
    50882: 'Bicarbonate', 50802: 'Base Excess', 51237: 'INR(PT)',
}

print(f"Culture items: {len(CULTURE_ITEMIDS)} items")
print(f"Antibiotic items:   {len(ANTIBIOTIC_ITEMIDS)} items")
print(f"Oxygen support items: {len(ALL_OXYGEN)} items")
print(f"Vasopressor items:   {len(VASOPRESSOR_ITEMIDS)} items")
print(f"Vitals+Lab:  {len(ADDITIONAL_VITAL_ITEMIDS) + len(LAB_ITEMIDS)} items")

Culture items: 8 items
Antibiotic items:   18 items
Oxygen support items: 13 items
Vasopressor items:   13 items
Vitals+Lab:  40 items


## 📌 Cell 3: First ICU Stay Extraction & Duration Filter

**Step 1 of 4 in cohort selection.**

We load `icustays.csv` and apply two filters:
1. **First ICU stay per patient**: If a patient visited the ICU multiple times, we only keep their first visit. This is done by ranking stays by admission time (`intime`) within each patient.
2. **Duration filter**: The stay must be between 12 hours and 10 days. Too short = not enough data. Too long = likely complex multi-system issues beyond sepsis.

In [3]:
print("=" * 60)
print("STEP 1: First ICU Stay Extraction & Filtering")
print("=" * 60)

icustays = pd.read_csv(f"{ICU_DIR}/icustays.csv")
print(f"  Total ICU stays: {len(icustays):,}")

# Select only first ICU stay per patient (by intime)
icustays['intime'] = pd.to_datetime(icustays['intime'])
icustays['outtime'] = pd.to_datetime(icustays['outtime'])
icustays['rn'] = icustays.groupby('subject_id')['intime'].rank(method='first').astype(int)
first_icu = icustays[icustays['rn'] == 1].copy()
print(f"  First ICU stay only: {len(first_icu):,}")

# Compute duration & filter (>12h, <10d)
first_icu['duration'] = first_icu['outtime'] - first_icu['intime']
first_icu = first_icu[
    (first_icu['duration'] > pd.Timedelta('12 hours')) &
    (first_icu['duration'] < pd.Timedelta('10 days'))
]
print(f"  12h < LOS < 10d: {len(first_icu):,}")

STEP 1: First ICU Stay Extraction & Filtering
  Total ICU stays: 94,458
  First ICU stay only: 65,366
  12h < LOS < 10d: 58,506


## 📌 Cell 4: Age Filter (18-65 Years)

**Step 2 of 4 in cohort selection.**

We merge with `patients.csv` to get each patient's age, then keep only those aged 18-65.
- Under 18: Pediatric patients have different physiology and treatment protocols
- Over 65: High prevalence of comorbidities that confound sepsis analysis

In [4]:
print("\n" + "=" * 60)
print("STEP 2: Age filter (18-65 years)")
print("=" * 60)

patients = pd.read_csv(f"{HOSP_DIR}/patients.csv")
first_icu = first_icu.merge(patients[['subject_id', 'anchor_age', 'gender', 'dod']],
                             on='subject_id', how='left')

first_icu = first_icu[
    (first_icu['anchor_age'] >= 18) &
    (first_icu['anchor_age'] <= 65)
]
print(f"  After 18-65 age filter: {len(first_icu):,}")


STEP 2: Age filter (18-65 years)
  After 18-65 age filter: 29,421


## 📌 Cell 5: Sepsis Cohort Definition (Antibiotics + Cultures ≤ 1 Hour)

**Step 3 of 4 — the most complex filter.**

A patient is "suspected of sepsis" if a doctor ordered BOTH:
1. **Antibiotics** (from `inputevents.csv` — IV antibiotic administration)
2. **Bacterial cultures** (from `procedureevents.csv` — e.g., blood culture, urine culture)

...and these occurred **within 1 hour of each other** during the ICU stay.

This is a standard clinical definition: if a doctor suspects infection badly enough to
start antibiotics AND order cultures simultaneously, sepsis is likely.

**Algorithm:**
1. Load all antibiotic events for patients who passed the age/duration filters
2. Load all culture events for the same patients
3. For each patient, check every (antibiotic, culture) pair
4. If any pair has `|antibiotic_time - culture_time| ≤ 3600 seconds` → patient has sepsis

In [5]:
print("\n" + "=" * 60)
print("STEP 3: Sepsis cohort (antibiotics + cultures ≤ 1 hour)")
print("=" * 60)

# Antibiotic administration records
inputevents = pd.read_csv(f"{ICU_DIR}/inputevents.csv")
abx_events = inputevents[inputevents['itemid'].isin(ANTIBIOTIC_ITEMIDS)].copy()
abx_events['starttime'] = pd.to_datetime(abx_events['starttime'])
abx_events['endtime'] = pd.to_datetime(abx_events['endtime'])
print(f"  Antibiotic administration records: {len(abx_events):,}")

# Culture test records
procedureevents = pd.read_csv(f"{ICU_DIR}/procedureevents.csv")
culture_events = procedureevents[procedureevents['itemid'].isin(CULTURE_ITEMIDS)].copy()
culture_events['starttime'] = pd.to_datetime(culture_events['starttime'])
print(f"  Culture records:   {len(culture_events):,}")

# Match only first ICU stay patients
valid_subjects = set(first_icu['subject_id'])
valid_stays = dict(zip(first_icu['stay_id'], zip(first_icu['intime'], first_icu['outtime'])))

abx_filtered = abx_events[
    (abx_events['subject_id'].isin(valid_subjects)) &
    (abx_events['stay_id'].isin(valid_stays.keys()))
].copy()
# Check if antibiotics are within ICU stay period
abx_valid = []
for _, row in abx_filtered.iterrows():
    sid = row['stay_id']
    if sid in valid_stays:
        intime, outtime = valid_stays[sid]
        if row['starttime'] >= intime and row['endtime'] <= outtime:
            abx_valid.append(row)
if abx_valid:
    abx_filtered = pd.DataFrame(abx_valid)
else:
    abx_filtered = pd.DataFrame()

culture_filtered = culture_events[
    (culture_events['subject_id'].isin(valid_subjects)) &
    (culture_events['stay_id'].isin(valid_stays.keys()))
].copy()
# Check if cultures are within ICU stay period
culture_valid = []
for _, row in culture_filtered.iterrows():
    sid = row['stay_id']
    if sid in valid_stays:
        intime, outtime = valid_stays[sid]
        if row['starttime'] >= intime and row['starttime'] <= outtime:
            culture_valid.append(row)
if culture_valid:
    culture_filtered = pd.DataFrame(culture_valid)
else:
    culture_filtered = pd.DataFrame()

# Match antibiotics + cultures within 1 hour
cohort_ids = set()
for subj in valid_subjects:
    abx_sub = abx_filtered[abx_filtered['subject_id'] == subj]
    cul_sub = culture_filtered[culture_filtered['subject_id'] == subj]
    if abx_sub.empty or cul_sub.empty:
        continue
    # cross-check: any pair within 1 hour?
    for _, a_row in abx_sub.iterrows():
        for _, c_row in cul_sub.iterrows():
            diff = abs((c_row['starttime'] - a_row['starttime']).total_seconds())
            if diff <= 3600:
                cohort_ids.add(subj)
                break
        if subj in cohort_ids:
            break

cohort = first_icu[first_icu['subject_id'].isin(cohort_ids)].copy()
print(f"\n✅ Final cohort: {len(cohort):,}patients")
print(f"   (first ICU stay + 12h-10d + age 18-65 + antibiotics·cultures ≤1hr)")

# Save key columns only
cohort_save = cohort[['subject_id', 'hadm_id', 'stay_id', 'intime', 'outtime',
                        'first_careunit', 'anchor_age', 'gender', 'dod']].copy()
cohort_save['duration_hours'] = (cohort_save['outtime'] - cohort_save['intime']).dt.total_seconds() / 3600
cohort_save.to_csv(f"{OUTPUT_DIR}/cohort.csv", index=False)
print(f"Save: cohort.csv")

cohort_stay_ids = set(cohort['stay_id'])
cohort_subject_ids = set(cohort['subject_id'])
cohort_hadm_ids = set(cohort['hadm_id'])


STEP 3: Sepsis cohort (antibiotics + cultures ≤ 1 hour)
  Antibiotic administration records: 522,949
  Culture records:   52,180

✅ Final cohort: 1,502patients
   (first ICU stay + 12h-10d + age 18-65 + antibiotics·cultures ≤1hr)
Save: cohort.csv


## 📌 Cell 6: Vital Signs Extraction (chartevents — 41.94 GB)

Now that we have our sepsis cohort (list of `stay_id` values), we extract their vital signs.

`chartevents.csv` is **41.94 GB** — far too large to load into memory at once.
We read it in chunks of 1 million rows, filter each chunk for our cohort, and collect results.

**This takes approximately 3-5 minutes** due to the file size.

In [6]:
print("\n" + "=" * 60)
print("STEP 4: Vital signs extraction (chartevents - 41.94 GB)")
print("       Takes approximately 4 minutes!")
print("=" * 60)

vital_ids = set(ADDITIONAL_VITAL_ITEMIDS.keys())

vitals_list = []
t0 = time.time()
for i, chunk in enumerate(pd.read_csv(
    f"{ICU_DIR}/chartevents.csv", chunksize=CHUNK_SIZE)):
    filtered = chunk[
        (chunk['stay_id'].isin(cohort_stay_ids)) &
        (chunk['itemid'].isin(vital_ids))
    ][['subject_id', 'hadm_id', 'stay_id', 'charttime',
       'itemid', 'valuenum', 'valueuom']]
    vitals_list.append(filtered)
    if (i + 1) % 50 == 0:
        elapsed = time.time() - t0
        rows = sum(len(v) for v in vitals_list)
        print(f"  {(i+1)*CHUNK_SIZE/1e6:.0f}Mrows | extracted: {rows:,} | {elapsed/60:.1f}min")

vitals = pd.concat(vitals_list, ignore_index=True)
vitals.to_csv(f"{OUTPUT_DIR}/vitals.csv", index=False)
print(f"✅ vitals.csv - {len(vitals):,}rows ({time.time()-t0:.0f}sec)")


STEP 4: Vital signs extraction (chartevents - 41.94 GB)
       Takes approximately 4 minutes!
  50Mrows | extracted: 177,242 | 0.5min
  100Mrows | extracted: 337,207 | 1.0min
  150Mrows | extracted: 477,181 | 1.4min
  200Mrows | extracted: 625,627 | 2.0min
  250Mrows | extracted: 763,977 | 2.5min
  300Mrows | extracted: 909,285 | 2.9min
  350Mrows | extracted: 1,067,325 | 3.4min
  400Mrows | extracted: 1,226,765 | 3.9min
✅ vitals.csv - 1,301,329rows (253sec)


## 📌 Cell 7: Lab Results Extraction (labevents — 18.4 GB)

Similar chunked reading for lab results.

**Important difference from chartevents**: `labevents.csv` does NOT have a `stay_id` column!
We filter by `subject_id` instead, and later in preprocessing we'll align lab times
with ICU stay windows using `hours_in >= 0`.

In [7]:
print("\n" + "=" * 60)
print("STEP 5: Lab results extraction (labevents - 18.4 GB)")
print("=" * 60)

lab_ids = set(LAB_ITEMIDS.keys())

labs_list = []
t0 = time.time()
for i, chunk in enumerate(pd.read_csv(
    f"{HOSP_DIR}/labevents.csv", chunksize=CHUNK_SIZE)):
    filtered = chunk[
        (chunk['subject_id'].isin(cohort_subject_ids)) &
        (chunk['itemid'].isin(lab_ids))
    ][['subject_id', 'hadm_id', 'charttime', 'itemid',
       'valuenum', 'valueuom', 'ref_range_lower', 'ref_range_upper', 'flag']]
    labs_list.append(filtered)
    if (i + 1) % 50 == 0:
        elapsed = time.time() - t0
        rows = sum(len(l) for l in labs_list)
        print(f"  {(i+1)*CHUNK_SIZE/1e6:.0f}Mrows | extracted: {rows:,} | {elapsed/60:.1f}min")

labs = pd.concat(labs_list, ignore_index=True)
labs.to_csv(f"{OUTPUT_DIR}/labs.csv", index=False)
print(f"✅ labs.csv - {len(labs):,}rows ({time.time()-t0:.0f}sec)")


STEP 5: Lab results extraction (labevents - 18.4 GB)
  50Mrows | extracted: 325,415 | 0.6min
  100Mrows | extracted: 675,549 | 1.2min
  150Mrows | extracted: 1,009,674 | 1.8min
✅ labs.csv - 1,052,508rows (119sec)


## 📌 Cell 8: Treatment Record Extraction

Extract three types of treatment data for our cohort:

1. **Inputevents** (vasopressors, IV fluids) — already loaded, just filter
2. **Procedureevents** (oxygen support, cultures) — filter for cohort
3. **Prescriptions** (antibiotics by keyword matching) — if available

In [8]:
print("\n" + "=" * 60)
print("STEP 6: Treatment record extraction")
print("=" * 60)

# Filter cohort from already-loaded inputevents
inputs_sepsis = inputevents[inputevents['stay_id'].isin(cohort_stay_ids)][
    ['subject_id', 'hadm_id', 'stay_id', 'starttime', 'endtime',
     'itemid', 'amount', 'amountuom', 'rate', 'rateuom',
     'ordercategoryname', 'patientweight', 'statusdescription']
]
inputs_sepsis.to_csv(f"{OUTPUT_DIR}/inputevents.csv", index=False)
print(f"✅ inputevents.csv - {len(inputs_sepsis):,}rows")
del inputevents

# Procedureevents (oxygen, cultures, etc.)
proc_sepsis = procedureevents[procedureevents['stay_id'].isin(cohort_stay_ids)]
proc_sepsis.to_csv(f"{OUTPUT_DIR}/procedureevents.csv", index=False)
print(f"✅ procedureevents.csv - {len(proc_sepsis):,}rows")
del procedureevents

# Prescriptions
print("\nprescriptions.csv...")
rx_list = []
for i, chunk in enumerate(pd.read_csv(
    f"{HOSP_DIR}/prescriptions.csv", chunksize=CHUNK_SIZE, low_memory=False)):
    filtered = chunk[chunk['hadm_id'].isin(cohort_hadm_ids)][
        ['subject_id', 'hadm_id', 'starttime', 'stoptime',
         'drug_type', 'drug', 'prod_strength', 'dose_val_rx',
         'dose_unit_rx', 'form_val_disp', 'form_unit_disp', 'route']
    ]
    rx_list.append(filtered)
rx_sepsis = pd.concat(rx_list, ignore_index=True)
rx_sepsis.to_csv(f"{OUTPUT_DIR}/prescriptions.csv", index=False)
print(f"✅ prescriptions.csv - {len(rx_sepsis):,}rows")
del rx_list, rx_sepsis


STEP 6: Treatment record extraction
✅ inputevents.csv - 331,093rows
✅ procedureevents.csv - 24,513rows

prescriptions.csv...
✅ prescriptions.csv - 204,848rows


## 📌 Cell 9: Supplementary Data

Additional files that support the analysis:
- **Outputevents**: Urine output (kidney function indicator)
- **Admissions**: Hospital-level data for the cohort
- **Patients**: Demographics for the cohort

In [9]:
print("\n" + "=" * 60)
print("STEP 7: Supplementary data")
print("=" * 60)

# Outputevents (urine output, etc.)
outputevents = pd.read_csv(f"{ICU_DIR}/outputevents.csv")
outputs_sepsis = outputevents[outputevents['stay_id'].isin(cohort_stay_ids)][
    ['subject_id', 'hadm_id', 'stay_id', 'charttime', 'itemid', 'value', 'valueuom']
]
outputs_sepsis.to_csv(f"{OUTPUT_DIR}/outputevents.csv", index=False)
print(f"✅ outputevents.csv - {len(outputs_sepsis):,}rows")
del outputevents, outputs_sepsis

# Admissions
admissions = pd.read_csv(f"{HOSP_DIR}/admissions.csv")
admissions_sepsis = admissions[admissions['hadm_id'].isin(cohort_hadm_ids)][
    ['subject_id', 'hadm_id', 'admittime', 'dischtime', 'deathtime',
     'admission_type', 'admission_location', 'discharge_location',
     'insurance', 'race', 'hospital_expire_flag']
]
admissions_sepsis.to_csv(f"{OUTPUT_DIR}/admissions.csv", index=False)
print(f"✅ admissions.csv - {len(admissions_sepsis):,}rows")

# Patients (cohort only)
patients_sepsis = patients[patients['subject_id'].isin(cohort_subject_ids)][
    ['subject_id', 'gender', 'anchor_age', 'anchor_year', 'anchor_year_group', 'dod']
]
patients_sepsis.to_csv(f"{OUTPUT_DIR}/patients.csv", index=False)
print(f"✅ patients.csv - {len(patients_sepsis):,}rows")


STEP 7: Supplementary data
✅ outputevents.csv - 124,878rows
✅ admissions.csv - 1,502rows
✅ patients.csv - 1,502rows


## 📌 Cell 10: Final Summary

Print extraction statistics and file sizes.

In [10]:
total_time = time.time() - start_time
print("\n" + "=" * 60)
print("✅ Extraction complete! (EXTRACTION COMPLETE)")
print("=" * 60)

total_size = 0
for f in sorted(os.listdir(OUTPUT_DIR)):
    filepath = os.path.join(OUTPUT_DIR, f)
    if os.path.isfile(filepath):
        size_mb = os.path.getsize(filepath) / (1024 * 1024)
        total_size += size_mb
        print(f"  {f:35s} {size_mb:>10.1f} MB")

print(f"\n  {'TOTAL':35s} {total_size:>10.1f} MB ({total_size/1024:.2f} GB)")
print(f"\n  Elapsed time:     {total_time/60:.1f}min")
print(f"  Sepsis cohort: {len(cohort):,}patients")
print(f"  Definition: first ICU stay + 12h-10d + age 18-65 + antibiotics·cultures ≤1hr")
print("=" * 60)
print("\n🔜 Next: 02_Data_Preprocessing.ipynb Run")


✅ Extraction complete! (EXTRACTION COMPLETE)
  admissions.csv                             0.2 MB
  cohort.csv                                 0.2 MB
  inputevents.csv                           44.7 MB
  labs.csv                                  68.4 MB
  outputevents.csv                           7.4 MB
  patients.csv                               0.1 MB
  prescriptions.csv                         21.8 MB
  procedureevents.csv                        4.4 MB
  vitals.csv                                79.1 MB

  TOTAL                                    226.1 MB (0.22 GB)

  Elapsed time:     7.7min
  Sepsis cohort: 1,502patients
  Definition: first ICU stay + 12h-10d + age 18-65 + antibiotics·cultures ≤1hr

🔜 Next: 02_Data_Preprocessing.ipynb Run
